# LandslideRisk-48: reference solution
Each day the crew may inspect any subset of the 12 new sites. An inspection costs the site's
`inspection_hours`; inspecting a site that has a landslide within 48 hours is worth 15 hours.
So the decision is: inspect site i on day d iff 15 * P(landslide) > inspection_hours.
Plan: (1) load data, (2) build features from each reading and *earlier* readings at the same
site, (3) estimate calibrated probabilities with folds grouped by site (every test site is
new), (4) compare decision policies on the net-value metric, (5) refit and plan the test days.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
SEED = 48
CATCH_VALUE = 15.0
TARGET = "landslide_risk_48h"
DATA = Path("./dataset/public")
OUT = Path("./working")
OUT.mkdir(parents=True, exist_ok=True)

# Queries (train.csv / test.csv): one row per day. train.csv lists the relevant sites ("none" if
# there are none). Readings (*_readings.csv): one row per site and day, with inspection_hours.
train_queries = pd.read_csv(DATA / "train.csv")
test_queries = pd.read_csv(DATA / "test.csv")
train = pd.read_csv(DATA / "train_readings.csv", parse_dates=["timestamp"])
test = pd.read_csv(DATA / "test_readings.csv", parse_dates=["timestamp"])
sample = pd.read_csv(DATA / "sample_submission.csv")
print(train_queries.shape, test_queries.shape, train.shape, test.shape, sample.shape)

# Per-reading relevance label: 1 if the site is listed in its day's location_ids.
relevant = {(q, int(s)) for q, sites in zip(train_queries.query_id, train_queries.location_ids)
            if sites != "none" for s in sites.split()}
train[TARGET] = [int((q, s) in relevant) for q, s in zip(train.query_id, train.location_id)]
print("train sites:", train.location_id.nunique(), "| test sites:", test.location_id.nunique(),
      "| overlap:", len(set(train.location_id) & set(test.location_id)))
print("positive rate:", round(train[TARGET].mean(), 4),
      "| inspection hours range:", train.inspection_hours.min(), "-", train.inspection_hours.max())

## Metric
Same as the grader: net value of the inspections divided by the net value of the perfect plan
(inspect exactly the relevant sites), floored at 0 (the value of inspecting nothing).

In [ ]:
def net_value_score(frame, inspect):
    rel = frame[TARGET].to_numpy() == 1
    hours = frame["inspection_hours"].to_numpy()
    inspect = np.asarray(inspect, dtype=bool)
    net = (CATCH_VALUE * (inspect & rel) - hours * inspect).sum()
    best = ((CATCH_VALUE - hours) * rel).sum()
    return max(0.0, net / best)

## Features
Each site's rows are processed separately, in timestamp order, and only the current and
earlier readings are used. Train and test sites never mix, so concatenating the two frames
only saves code. `location_id` is not a model feature because every test site is new.

In [ ]:
RAIN = ["rainfall_1h_mm", "rainfall_6h_mm", "rainfall_24h_mm", "rainfall_72h_mm"]

def build_features(df):
    df = df.sort_values(["location_id", "timestamp"]).copy()
    g = df.groupby("location_id")

    # Fill small sensor gaps with the site's previous reading (past only).
    for c in RAIN + ["soil_moisture", "vegetation_index", "temperature_c"]:
        df[c + "_ffill"] = g[c].ffill()

    doy = df.timestamp.dt.dayofyear
    df["doy_sin"], df["doy_cos"] = np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25)
    df["aspect_sin"], df["aspect_cos"] = np.sin(np.radians(df.aspect_deg)), np.cos(np.radians(df.aspect_deg))

    r1, r6, r24, r72 = (df[c + "_ffill"] for c in RAIN)
    df["rain_1_6h"], df["rain_6_24h"], df["rain_24_72h"] = r6 - r1, r24 - r6, r72 - r24
    df["rain_24_over_72"] = r24 / (r72 + 1.0)
    df["slope_x_moist"] = df.slope_deg * df.soil_moisture_ffill
    df["slope_x_rain72"] = df.slope_deg * r72
    df["moist_x_rain72"] = df.soil_moisture_ffill * r72
    df["sin_slope"] = np.sin(np.radians(df.slope_deg))

    # Antecedent wetness from earlier days at the same site.
    g = df.groupby("location_id")
    for c in ["rainfall_72h_mm_ffill", "soil_moisture_ffill"]:
        df[c + "_lag1"] = g[c].shift(1)
        df[c + "_lag3"] = g[c].shift(3)
        df[c + "_d1"] = df[c] - df[c + "_lag1"]
    df["rain24_sum7d"] = g["rainfall_24h_mm_ffill"].transform(lambda s: s.rolling(7, min_periods=1).sum())
    df["rain24_sum30d"] = g["rainfall_24h_mm_ffill"].transform(lambda s: s.rolling(30, min_periods=1).sum())
    df["moist_mean7d"] = g["soil_moisture_ffill"].transform(lambda s: s.rolling(7, min_periods=1).mean())

    df["soil_type"] = df.soil_type.astype(pd.CategoricalDtype(["sandy", "loam", "clay", "silt", "rocky", "peat"]))
    return df

full = build_features(pd.concat([train.assign(_split="train"), test.assign(_split="test")], ignore_index=True))
tr = full[full._split == "train"].reset_index(drop=True)
te = full[full._split == "test"].reset_index(drop=True)

BASE = ["elevation_m", "slope_deg", "aspect_deg", "soil_moisture", "soil_type", "vegetation_index",
        *RAIN, "temperature_c"]
ENGINEERED = BASE + ["doy_sin", "doy_cos", "aspect_sin", "aspect_cos", "rain_1_6h", "rain_6_24h",
        "rain_24_72h", "rain_24_over_72", "slope_x_moist", "slope_x_rain72", "moist_x_rain72", "sin_slope",
        "rainfall_72h_mm_ffill_lag1", "rainfall_72h_mm_ffill_lag3", "rainfall_72h_mm_ffill_d1",
        "soil_moisture_ffill_lag1", "soil_moisture_ffill_lag3", "soil_moisture_ffill_d1",
        "rain24_sum7d", "rain24_sum30d", "moist_mean7d"]

## Validation that mirrors the test set
Three folds grouped by site: each validation fold holds ~13 unseen sites, close to the 12 test
sites. Out-of-fold (OOF) probabilities let every decision policy be scored on unseen sites.

In [ ]:
folds = list(GroupKFold(n_splits=3).split(tr, groups=tr.location_id))

LGB_PARAMS = dict(objective="binary", learning_rate=0.02, n_estimators=2000, num_leaves=4,
                  min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
                  reg_lambda=5.0, random_state=SEED, verbose=-1)

def make_lr(features):
    num = [c for c in features if c != "soil_type"]
    prep = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("soil", OneHotEncoder(handle_unknown="ignore"), ["soil_type"]),
    ])
    return make_pipeline(prep, LogisticRegression(max_iter=3000))

def oof_lr(features):
    oof = np.zeros(len(tr))
    for f, v in folds:
        m = make_lr(features).fit(tr.loc[f, features], tr.loc[f, TARGET])
        oof[v] = m.predict_proba(tr.loc[v, features])[:, 1]
    return oof

best_iters = []
def oof_lgb(features):
    oof = np.zeros(len(tr))
    for f, v in folds:
        m = lgb.LGBMClassifier(**LGB_PARAMS)
        m.fit(tr.loc[f, features], tr.loc[f, TARGET], eval_set=[(tr.loc[v, features], tr.loc[v, TARGET])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        best_iters.append(m.best_iteration_)
        oof[v] = m.predict_proba(tr.loc[v, features])[:, 1]
    return oof

def fold_score(inspect):
    inspect = np.asarray(inspect, dtype=bool)
    return float(np.mean([net_value_score(tr.loc[v], inspect[v]) for _, v in folds]))

def logloss(p):
    p = np.clip(p, 1e-6, 1 - 1e-6); y = tr[TARGET].to_numpy()
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

oof = {"LR raw": oof_lr(BASE), "LR engineered": oof_lr(ENGINEERED), "LightGBM engineered": oof_lgb(ENGINEERED)}
oof["blend LR raw + LightGBM"] = 0.5 * oof["LR raw"] + 0.5 * oof["LightGBM engineered"]
for name, p in oof.items():
    print(f"{name:26s} OOF log loss {logloss(p):.4f} | mean p {p.mean():.4f} vs rate {tr[TARGET].mean():.4f}")

### Decision policies
Fixed top-K ignores both how risky a day is and what each inspection costs. The decision rule
`15 * p > inspection_hours` uses both, and only works if the probabilities are calibrated.

In [ ]:
hours = tr.inspection_hours.to_numpy()
day_rank = lambda s: tr.assign(s=s).groupby("query_id")["s"].rank(ascending=False, method="first").to_numpy()
rain = tr.rainfall_72h_mm_ffill.fillna(0).to_numpy()
results = {
    "inspect nothing": fold_score(np.zeros(len(tr))),
    "inspect everything": fold_score(np.ones(len(tr))),
    "top-1 by rainfall_72h every day": fold_score(day_rank(rain) <= 1),
    "top-3 by LR raw every day": fold_score(day_rank(oof["LR raw"]) <= 3),
    "top-1 by LR raw every day": fold_score(day_rank(oof["LR raw"]) <= 1),
    "15*p > mean hours (ignores site cost), LR raw": fold_score(CATCH_VALUE * oof["LR raw"] > hours.mean()),
}
for name, p in oof.items():
    results[f"15*p > hours, {name}"] = fold_score(CATCH_VALUE * p > hours)
summary = pd.Series(results, name="site-grouped net value score (higher is better)").round(4)
print(summary.to_string())

### Choose the model and check calibration of the decision threshold
A multiplier `alpha` on the threshold (`15 * p > alpha * hours`) would be ~1 for calibrated
probabilities; tuning it on the OOF predictions guards against residual miscalibration.

In [ ]:
policy_scores = {k: v for k, v in results.items() if k.startswith("15*p > hours")}
best_name = max(policy_scores, key=policy_scores.get).split(", ", 1)[1]
alphas = np.round(np.arange(0.6, 1.61, 0.05), 2)
alpha_scores = {a: fold_score(CATCH_VALUE * oof[best_name] > a * hours) for a in alphas}
alpha = max(alpha_scores, key=alpha_scores.get)
print("best probability model:", best_name)
print("alpha scan:", {a: round(s, 4) for a, s in alpha_scores.items() if a in (0.6, 0.8, 1.0, 1.2, 1.4, 1.6)})
print(f"chosen alpha = {alpha} (score {alpha_scores[alpha]:.4f} vs {alpha_scores[1.0]:.4f} at alpha = 1)")

## Final plan
Refit the chosen probability model on all training sites, then inspect every test site whose
expected catch value exceeds its (alpha-scaled) inspection hours. Days with no such site get "none".

In [ ]:
def fit_predict(name):
    if name == "LR raw":
        return make_lr(BASE).fit(tr[BASE], tr[TARGET]).predict_proba(te[BASE])[:, 1]
    if name == "LR engineered":
        return make_lr(ENGINEERED).fit(tr[ENGINEERED], tr[TARGET]).predict_proba(te[ENGINEERED])[:, 1]
    n = int(np.mean(best_iters) * 1.2)
    p = np.zeros(len(te))
    for s in range(5):
        m = lgb.LGBMClassifier(**{**LGB_PARAMS, "n_estimators": n, "random_state": SEED + s})
        p += m.fit(tr[ENGINEERED], tr[TARGET]).predict_proba(te[ENGINEERED])[:, 1] / 5
    if name == "LightGBM engineered":
        return p
    return 0.5 * p + 0.5 * fit_predict("LR raw")

te["p"] = fit_predict(best_name)
te["inspect"] = CATCH_VALUE * te.p > alpha * te.inspection_hours
plan = (te[te.inspect].sort_values(["query_id", "p"], ascending=[True, False])
          .groupby("query_id").location_id.agg(lambda s: " ".join(map(str, s))))
submission = test_queries[["query_id"]].assign(location_ids=lambda d: d.query_id.map(plan).fillna("none"))

assert len(submission) == len(sample) and set(submission.query_id) == set(sample.query_id)
print("inspections per day:", round(te.inspect.sum() / len(submission), 2),
      "| days with no inspection:", int((submission.location_ids == "none").sum()))
submission.to_csv(OUT / "submission.csv", index=False)
submission.head()